In [ ]:
!pip install gensim nltk spacy matplotlib pandas numpy --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 71.3 MB/s eta 0:00:00


In [ ]:
from joblib import Parallel, delayed
from spacy.lang.en.stop_words import STOP_WORDS
import os

import spacy

# Load spaCy model
print("⏳ Loading spaCy model...")
try:
    nlp = spacy.load('en_core_web_sm')
    print("✅ spaCy model loaded successfully!")
except OSError:
    print("⚠️  Model not found. Downloading 'en_core_web_sm'...")
    import subprocess
    subprocess.run(['python', '-m', 'spacy', 'download', 'en_core_web_sm'])
    nlp = spacy.load('en_core_web_sm')
    print("✅ spaCy model downloaded and loaded!")


# Define stop words
print("⏳ Loading stop words...")
stop_words = STOP_WORDS
# You can also add custom stop words if needed:
# stop_words = STOP_WORDS.union({'custom', 'words', 'here'})
print(f"✅ Stop words loaded! Total: {len(stop_words)} words")



⏳ Loading spaCy model...
✅ spaCy model loaded successfully!
⏳ Loading stop words...
✅ Stop words loaded! Total: 326 words


In [ ]:
import pandas as pd
import numpy as np
import re
import os
import time

# Gensim for LDA
from gensim.models.ldamulticore import LdaMulticore
from gensim.corpora.dictionary import Dictionary
from gensim.models import CoherenceModel

# Preprocessing
import nltk
from nltk.corpus import stopwords
import spacy

# Visualization and Utilities
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

# Download NLTK resources
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
!python -m spacy download en_core_web_sm --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 98.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
# Load the dataset
TEXT_COLUMN = 'sentence'

url = 'https://docs.google.com/spreadsheets/d/e/2PACX-1vQPPsr1--5VLu4j2NgBvzKG29Ny6ci5VZ5bTEoSDnakEtjBrnews05sOOeRtAxkONLCbVghNWDHKLq1/pub?output=csv'
df = pd.read_csv(url, encoding='utf-8')

# Display the first few rows of the DataFrame
display(df.head())

,review_id,sentence_num,sentiment,sentence
0,1,2,2,This exerciser certainly checked all the boxes...
1,1,3,2,The seat position is extremely comfortable and...
2,1,4,2,Provides a good range of resistance from super...
3,1,5,2,And the resistance bands and the multiple posi...
4,1,6,2,There are so many different muscles you are ab...


In [ ]:
#Filter by sentiment
df_positive = df[df['sentiment'] == 2].copy()
df_negative = df[df['sentiment'] == 0].copy()

In [ ]:
def _process_texts_sequential(text_list):
    """
    Process a list of texts sequentially.
    (Lowercase, Punctuation, Stop Words, Lemmatization)
    """
    processed_texts = []
    # Clean punctuation and lowercase first
    raw_texts = [re.sub(r'[^a-z\s]', '', str(text).lower()) for text in text_list]

    # Process texts sequentially using nlp.pipe for efficiency
    for doc in nlp.pipe(raw_texts, batch_size=5000):
        tokens = [
            token.lemma_ for token in doc
            if token.lemma_ not in stop_words and len(token.lemma_) > 2
        ]
        processed_texts.append(tokens)
    return processed_texts

def run_sequential_preprocessing(text_list):
    """
    Wrapper function for sequential preprocessing.
    """
    print(f"Starting sequential preprocessing for {len(text_list)} documents...")
    start_time = time.time()

    texts_for_lda = _process_texts_sequential(text_list)

    end_time = time.time()
    print(f"Preprocessing complete in {end_time - start_time:.2f} seconds.")
    return texts_for_lda

# --- LDA Pipeline (Sequential) ---

def _train_and_score(num_topics, corpus, dictionary, processed_texts):
    """
    Helper function to train one LDA model and return its coherence score.
    """
    print(f"Testing K={num_topics}...")

    model = LdaMulticore(
        corpus=corpus, id2word=dictionary, num_topics=num_topics,
        workers=1, # Run sequentially
        chunksize=20000,
        passes=5,
        random_state=100
    )

    coherence_model = CoherenceModel(
        model=model, texts=processed_texts,
        dictionary=dictionary, coherence='c_v'
    )
    coherence = coherence_model.get_coherence()
    print(f"K={num_topics} Coherence: {coherence:.4f}")
    return (num_topics, coherence, model)

def run_lda_pipeline_sequential(processed_texts, label):
    """
    Runs the full LDA pipeline 100% sequentially.
    """
    print(f"\n--- 1. Creating Dictionary & Corpus for '{label}' Model ---")
    dictionary = Dictionary(processed_texts)
    dictionary.filter_extremes(no_below=30, no_above=0.5)
    print(f"Vocabulary size for '{label}': {len(dictionary)}")

    corpus = [dictionary.doc2bow(text) for text in processed_texts]
    if not corpus:
        print(f"Corpus for '{label}' is empty. Skipping.")
        return None, None, None, None, None

    # --- 2. Find Optimal K (SEQUENTIAL) ---
    print(f"\n--- 2. Finding Optimal K for '{label}' Model (Sequentially) ---")
    start_time = time.time()
    K_RANGE = range(2, 33, 2) # Test K=2, 4, 6, 8, ..., 30, 32

    results = []
    all_k_results = []  # Store ALL results with models
    all_models = {}  # Store all models by K

    # Run all coherence tests in a simple for loop
    for k in K_RANGE:
        result = _train_and_score(k, corpus, dictionary, processed_texts)
        results.append(result)

        # Store k, coherence, and model
        all_k_results.append({
            'k': result[0],
            'coherence': result[1]
        })
        all_models[result[0]] = result[2]  # Store the model

        print(f"  K={k}: Coherence={result[1]:.4f}")

    end_time = time.time()
    print(f"Coherence calculation finished in {end_time - start_time:.2f} seconds.")

    # Find the best result
    best_result = max(results, key=lambda item: item[1])
    best_k = best_result[0]
    best_coherence = best_result[1]
    final_model = best_result[2]

    print(f"Optimal K for '{label}': {best_k} (Coherence: {best_coherence:.4f})")

    # --- 3. Show Topics from Best Model ---
    print(f"\n--- ✅ TOP TOPICS for '{label}' Model (K={best_k}) ---")
    topics = final_model.print_topics(num_words=10)
    for topic in topics:
        print(topic)

    return final_model, dictionary, corpus, all_k_results, all_models

In [ ]:
import pickle
import json
from datetime import datetime
import time

# Add logging at the start
print("\n" + "="*80)
print("🚀 LDA TOPIC MODELING PIPELINE STARTED")
print(f"⏰ Start Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*80)

print("\n📋 PIPELINE OVERVIEW:")
print("  Step 1: Negative Sentiment Processing & Modeling")
print("  Step 2: Positive Sentiment Processing & Modeling")
print("  Step 3: Saving Binary Models")
print("  Step 4: Generating Human-Readable Reports")
print("  Step 5: Creating Complete JSON Outputs (ALL K values)")

# ========== STEP 1: NEGATIVE SENTIMENT ==========
print("\n" + "="*80)
print("🔴 STEP 1/2: NEGATIVE SENTIMENT PIPELINE")
print("="*80)
print(f"📊 Dataset size: {len(df_negative)} reviews")
print("⏳ Status: Starting preprocessing...")

start_time_neg = datetime.now()
negative_texts_processed = run_sequential_preprocessing(df_negative[TEXT_COLUMN].tolist())
print(f"✅ Preprocessing complete! Processed {len(negative_texts_processed)} texts")
print(f"⏱️  Preprocessing time: {(datetime.now() - start_time_neg).total_seconds():.2f} seconds")

print("\n⏳ Status: Starting LDA modeling (testing K=2 to K=32)...")
print("   This will test 16 different K values - progress will be shown below:")
model_start_neg = datetime.now()
model_neg, dict_neg, corpus_neg, all_k_results_neg, all_models_neg = run_lda_pipeline_sequential(negative_texts_processed, "Negative")
print(f"✅ Negative model complete!")
print(f"⏱️  Modeling time: {(datetime.now() - model_start_neg).total_seconds():.2f} seconds")
print(f"⏱️  Total negative pipeline time: {(datetime.now() - start_time_neg).total_seconds():.2f} seconds")

# ========== STEP 2: POSITIVE SENTIMENT ==========
print("\n" + "="*80)
print("🟢 STEP 2/2: POSITIVE SENTIMENT PIPELINE")
print("="*80)
print(f"📊 Dataset size: {len(df_positive)} reviews")
print("⏳ Status: Starting preprocessing...")

start_time_pos = datetime.now()
positive_texts_processed = run_sequential_preprocessing(df_positive[TEXT_COLUMN].tolist())
print(f"✅ Preprocessing complete! Processed {len(positive_texts_processed)} texts")
print(f"⏱️  Preprocessing time: {(datetime.now() - start_time_pos).total_seconds():.2f} seconds")

print("\n⏳ Status: Starting LDA modeling (testing K=2 to K=32)...")
print("   This will test 16 different K values - progress will be shown below:")
model_start_pos = datetime.now()
model_pos, dict_pos, corpus_pos, all_k_results_pos, all_models_pos = run_lda_pipeline_sequential(positive_texts_processed, "Positive")
print(f"✅ Positive model complete!")
print(f"⏱️  Modeling time: {(datetime.now() - model_start_pos).total_seconds():.2f} seconds")
print(f"⏱️  Total positive pipeline time: {(datetime.now() - start_time_pos).total_seconds():.2f} seconds")

# ========== STEP 3: SAVE BINARY MODELS ==========
print("\n" + "="*80)
print("💾 STEP 3/5: SAVING BINARY MODELS")
print("="*80)

save_start = datetime.now()
files_saved = []

if model_neg is not None:
    print("⏳ Saving negative sentiment best model...")
    model_neg.save('lda_model_negative.model')
    files_saved.append('lda_model_negative.model')
    print("  ✓ Best model saved")

    dict_neg.save('dictionary_negative.dict')
    files_saved.append('dictionary_negative.dict')
    print("  ✓ Dictionary saved")

    with open('corpus_negative.pkl', 'wb') as f:
        pickle.dump(corpus_neg, f)
    files_saved.append('corpus_negative.pkl')
    print("  ✓ Corpus saved")

    with open('processed_texts_negative.pkl', 'wb') as f:
        pickle.dump(negative_texts_processed, f)
    files_saved.append('processed_texts_negative.pkl')
    print("  ✓ Processed texts saved")

    # Save all models
    print("⏳ Saving ALL negative sentiment models...")
    with open('all_models_negative.pkl', 'wb') as f:
        pickle.dump(all_models_neg, f)
    files_saved.append('all_models_negative.pkl')
    print("  ✓ All models saved")

    print("✅ All negative sentiment files saved!")
else:
    print("⚠️  Negative model is None, skipping save")

if model_pos is not None:
    print("\n⏳ Saving positive sentiment best model...")
    model_pos.save('lda_model_positive.model')
    files_saved.append('lda_model_positive.model')
    print("  ✓ Best model saved")

    dict_pos.save('dictionary_positive.dict')
    files_saved.append('dictionary_positive.dict')
    print("  ✓ Dictionary saved")

    with open('corpus_positive.pkl', 'wb') as f:
        pickle.dump(corpus_pos, f)
    files_saved.append('corpus_positive.pkl')
    print("  ✓ Corpus saved")

    with open('processed_texts_positive.pkl', 'wb') as f:
        pickle.dump(positive_texts_processed, f)
    files_saved.append('processed_texts_positive.pkl')
    print("  ✓ Processed texts saved")

    # Save all models
    print("⏳ Saving ALL positive sentiment models...")
    with open('all_models_positive.pkl', 'wb') as f:
        pickle.dump(all_models_pos, f)
    files_saved.append('all_models_positive.pkl')
    print("  ✓ All models saved")

    print("✅ All positive sentiment files saved!")
else:
    print("⚠️  Positive model is None, skipping save")

print(f"⏱️  Binary save time: {(datetime.now() - save_start).total_seconds():.2f} seconds")

# ========== STEP 4: GENERATE COMPREHENSIVE TEXT REPORT ==========
print("\n" + "="*80)
print("📝 STEP 4/5: GENERATING COMPREHENSIVE TEXT REPORT")
print("="*80)

report_start = datetime.now()
print("⏳ Writing detailed analysis to lda_complete_results.txt...")

with open('lda_complete_results.txt', 'w', encoding='utf-8') as f:
    f.write("="*100 + "\n")
    f.write("COMPLETE LDA TOPIC MODELING RESULTS\n")
    f.write(f"Generated on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write("="*100 + "\n\n")

    # ========== NEGATIVE SENTIMENT ==========
    if model_neg is not None:
        print("  ⏳ Processing negative sentiment results...")
        f.write("\n" + "🔴 "*25 + "\n")
        f.write("NEGATIVE SENTIMENT ANALYSIS\n")
        f.write("🔴 "*25 + "\n\n")

        # K Selection Results
        f.write("K SELECTION RESULTS (All tested values):\n")
        f.write("-"*100 + "\n")
        f.write(f"{'K':<10} {'Coherence Score':<20}\n")
        f.write("-"*100 + "\n")
        for result in all_k_results_neg:
            marker = " ⭐ SELECTED" if result['k'] == model_neg.num_topics else ""
            f.write(f"{result['k']:<10} {result['coherence']:<20.6f}{marker}\n")
        f.write("\n")

        f.write("SUMMARY STATISTICS (BEST MODEL):\n")
        f.write("-"*100 + "\n")
        f.write(f"  • Optimal K (Selected): {model_neg.num_topics}\n")
        f.write(f"  • Vocabulary Size: {len(dict_neg)} unique tokens\n")
        f.write(f"  • Number of Documents: {len(corpus_neg)} reviews\n")
        f.write(f"  • Total Processed Texts: {len(negative_texts_processed)} documents\n")
        f.write(f"  • Model Perplexity: {model_neg.log_perplexity(corpus_neg):.4f}\n\n")

        # All topics with detailed information (BEST MODEL ONLY)
        f.write("DETAILED TOPICS (BEST MODEL ONLY - See JSON for all K values):\n")
        f.write("="*100 + "\n\n")

        for topic_id in range(model_neg.num_topics):
            print(f"    ⏳ Writing Topic {topic_id}/{model_neg.num_topics}...")
            f.write(f"TOPIC {topic_id}\n")
            f.write("-"*100 + "\n")

            # Get top 20 words for this topic
            topic_words = model_neg.show_topic(topic_id, topn=20)

            # Format as readable list
            f.write("Top 20 words (word: probability):\n")
            for word, prob in topic_words:
                f.write(f"  • {word:20s} : {prob:.6f}\n")

            # Raw topic string
            f.write("\nRaw topic representation:\n")
            f.write(f"  {model_neg.print_topic(topic_id, topn=20)}\n")
            f.write("\n" + "="*100 + "\n\n")

        # Topic distribution statistics
        print("  ⏳ Calculating topic distributions...")
        f.write("TOPIC DISTRIBUTION ACROSS DOCUMENTS:\n")
        f.write("-"*100 + "\n")
        topic_counts = {i: 0 for i in range(model_neg.num_topics)}

        for doc_bow in corpus_neg:
            doc_topics = model_neg.get_document_topics(doc_bow)
            if doc_topics:
                dominant_topic = max(doc_topics, key=lambda x: x[1])[0]
                topic_counts[dominant_topic] += 1

        for topic_id, count in topic_counts.items():
            percentage = (count / len(corpus_neg) * 100) if corpus_neg else 0
            f.write(f"  Topic {topic_id:2d}: {count:5d} documents ({percentage:5.2f}%)\n")

        f.write("\n\n")
        print("  ✅ Negative sentiment results written")

    # ========== POSITIVE SENTIMENT ==========
    if model_pos is not None:
        print("  ⏳ Processing positive sentiment results...")
        f.write("\n" + "🟢 "*25 + "\n")
        f.write("POSITIVE SENTIMENT ANALYSIS\n")
        f.write("🟢 "*25 + "\n\n")

        # K Selection Results
        f.write("K SELECTION RESULTS (All tested values):\n")
        f.write("-"*100 + "\n")
        f.write(f"{'K':<10} {'Coherence Score':<20}\n")
        f.write("-"*100 + "\n")
        for result in all_k_results_pos:
            marker = " ⭐ SELECTED" if result['k'] == model_pos.num_topics else ""
            f.write(f"{result['k']:<10} {result['coherence']:<20.6f}{marker}\n")
        f.write("\n")

        f.write("SUMMARY STATISTICS (BEST MODEL):\n")
        f.write("-"*100 + "\n")
        f.write(f"  • Optimal K (Selected): {model_pos.num_topics}\n")
        f.write(f"  • Vocabulary Size: {len(dict_pos)} unique tokens\n")
        f.write(f"  • Number of Documents: {len(corpus_pos)} reviews\n")
        f.write(f"  • Total Processed Texts: {len(positive_texts_processed)} documents\n")
        f.write(f"  • Model Perplexity: {model_pos.log_perplexity(corpus_pos):.4f}\n\n")

        # All topics with detailed information (BEST MODEL ONLY)
        f.write("DETAILED TOPICS (BEST MODEL ONLY - See JSON for all K values):\n")
        f.write("="*100 + "\n\n")

        for topic_id in range(model_pos.num_topics):
            print(f"    ⏳ Writing Topic {topic_id}/{model_pos.num_topics}...")
            f.write(f"TOPIC {topic_id}\n")
            f.write("-"*100 + "\n")

            # Get top 20 words for this topic
            topic_words = model_pos.show_topic(topic_id, topn=20)

            # Format as readable list
            f.write("Top 20 words (word: probability):\n")
            for word, prob in topic_words:
                f.write(f"  • {word:20s} : {prob:.6f}\n")

            # Raw topic string
            f.write("\nRaw topic representation:\n")
            f.write(f"  {model_pos.print_topic(topic_id, topn=20)}\n")
            f.write("\n" + "="*100 + "\n\n")

        # Topic distribution statistics
        print("  ⏳ Calculating topic distributions...")
        f.write("TOPIC DISTRIBUTION ACROSS DOCUMENTS:\n")
        f.write("-"*100 + "\n")
        topic_counts = {i: 0 for i in range(model_pos.num_topics)}

        for doc_bow in corpus_pos:
            doc_topics = model_pos.get_document_topics(doc_bow)
            if doc_topics:
                dominant_topic = max(doc_topics, key=lambda x: x[1])[0]
                topic_counts[dominant_topic] += 1

        for topic_id, count in topic_counts.items():
            percentage = (count / len(corpus_pos) * 100) if corpus_pos else 0
            f.write(f"  Topic {topic_id:2d}: {count:5d} documents ({percentage:5.2f}%)\n")

        f.write("\n\n")
        print("  ✅ Positive sentiment results written")

    # ========== COMPARISON ==========
    print("  ⏳ Writing comparison summary...")
    f.write("\n" + "📊 "*25 + "\n")
    f.write("COMPARISON SUMMARY\n")
    f.write("📊 "*25 + "\n\n")

    if model_neg and model_pos:
        f.write(f"{'Metric':<30} {'Negative':<20} {'Positive':<20}\n")
        f.write("-"*100 + "\n")
        f.write(f"{'Number of Topics':<30} {model_neg.num_topics:<20} {model_pos.num_topics:<20}\n")
        f.write(f"{'Vocabulary Size':<30} {len(dict_neg):<20} {len(dict_pos):<20}\n")
        f.write(f"{'Number of Documents':<30} {len(corpus_neg):<20} {len(corpus_pos):<20}\n")
        f.write(f"{'Perplexity':<30} {model_neg.log_perplexity(corpus_neg):<20.4f} {model_pos.log_perplexity(corpus_pos):<20.4f}\n")

    f.write("\n" + "="*100 + "\n")
    f.write("END OF REPORT\n")
    f.write("="*100 + "\n")

print("✅ Text report complete: lda_complete_results.txt")
print(f"⏱️  Report generation time: {(datetime.now() - report_start).total_seconds():.2f} seconds")

# ========== STEP 5: GENERATE COMPLETE JSON WITH ALL K VALUES ==========
print("\n" + "="*80)
print("📊 STEP 5/5: GENERATING COMPLETE JSON OUTPUT (ALL K VALUES)")
print("="*80)

json_start = datetime.now()
print("⏳ Creating JSON structure with ALL K values...")

json_results = {
    'metadata': {
        'generated_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'k_range_tested': '2-32 (step 2)',
        'note': 'This file contains topics for ALL tested K values'
    },
    'negative': {
        'optimal_k': None,
        'all_k_models': []
    },
    'positive': {
        'optimal_k': None,
        'all_k_models': []
    }
}

if model_neg is not None and all_models_neg is not None:
    print("  ⏳ Extracting ALL negative sentiment models...")
    json_results['negative']['optimal_k'] = model_neg.num_topics

    for k_value in sorted(all_models_neg.keys()):
        print(f"    ⏳ Processing K={k_value}...")
        current_model = all_models_neg[k_value]
        coherence_score = next(r['coherence'] for r in all_k_results_neg if r['k'] == k_value)

        k_data = {
            'k': k_value,
            'coherence': coherence_score,
            'is_optimal': k_value == model_neg.num_topics,
            'perplexity': current_model.log_perplexity(corpus_neg),
            'topics': []
        }

        for topic_id in range(current_model.num_topics):
            topic_words = current_model.show_topic(topic_id, topn=20)
            k_data['topics'].append({
                'topic_id': topic_id,
                'words': [{'word': word, 'probability': float(prob)} for word, prob in topic_words]
            })

        json_results['negative']['all_k_models'].append(k_data)

    print(f"  ✅ Extracted topics for {len(all_models_neg)} different K values (negative)")

if model_pos is not None and all_models_pos is not None:
    print("  ⏳ Extracting ALL positive sentiment models...")
    json_results['positive']['optimal_k'] = model_pos.num_topics

    for k_value in sorted(all_models_pos.keys()):
        print(f"    ⏳ Processing K={k_value}...")
        current_model = all_models_pos[k_value]
        coherence_score = next(r['coherence'] for r in all_k_results_pos if r['k'] == k_value)

        k_data = {
            'k': k_value,
            'coherence': coherence_score,
            'is_optimal': k_value == model_pos.num_topics,
            'perplexity': current_model.log_perplexity(corpus_pos),
            'topics': []
        }

        for topic_id in range(current_model.num_topics):
            topic_words = current_model.show_topic(topic_id, topn=20)
            k_data['topics'].append({
                'topic_id': topic_id,
                'words': [{'word': word, 'probability': float(prob)} for word, prob in topic_words]
            })

        json_results['positive']['all_k_models'].append(k_data)

    print(f"  ✅ Extracted topics for {len(all_models_pos)} different K values (positive)")

print("⏳ Writing complete JSON file...")
with open('lda_all_k_results_complete.json', 'w', encoding='utf-8') as f:
    json.dump(json_results, f, indent=2, ensure_ascii=False)

print("✅ Complete JSON output: lda_all_k_results_complete.json")
print(f"⏱️  JSON generation time: {(datetime.now() - json_start).total_seconds():.2f} seconds")

# ========== FINAL SUMMARY ==========
total_time = (datetime.now() - start_time_neg).total_seconds()

print("\n" + "="*80)
print("🎉 PIPELINE COMPLETED SUCCESSFULLY!")
print("="*80)

print("\n⏱️  TIMING SUMMARY:")
print(f"  • Negative pipeline: {(datetime.now() - start_time_neg).total_seconds():.2f}s")
print(f"  • Positive pipeline: {(datetime.now() - start_time_pos).total_seconds():.2f}s")
print(f"  • File saving & reports: {(datetime.now() - save_start).total_seconds():.2f}s")
print(f"  • Total elapsed time: {total_time:.2f}s ({total_time/60:.2f} minutes)")

print("\n📁 FILES GENERATED:")
print("\n  🤖 Binary Models (for reloading):")
for i, fname in enumerate(files_saved, 1):
    print(f"     {i}. {fname}")

print("\n  📖 Human-Readable Reports:")
print("     1. lda_complete_results.txt (Best model details)")
print("     2. lda_all_k_results_complete.json ⭐ (ALL K values with topics)")

print("\n✨ All Done!")
print("   • For best model details: lda_complete_results.txt")
print("   • For ALL K values with topics: lda_all_k_results_complete.json")
print("="*80 + "\n")


🚀 LDA TOPIC MODELING PIPELINE STARTED
⏰ Start Time: 2025-11-13 04:02:32

📋 PIPELINE OVERVIEW:
  Step 1: Negative Sentiment Processing & Modeling
  Step 2: Positive Sentiment Processing & Modeling
  Step 3: Saving Binary Models
  Step 4: Generating Human-Readable Reports
  Step 5: Creating Complete JSON Outputs (ALL K values)

🔴 STEP 1/2: NEGATIVE SENTIMENT PIPELINE
📊 Dataset size: 110398 reviews
⏳ Status: Starting preprocessing...
Starting sequential preprocessing for 110398 documents...
Preprocessing complete in 289.44 seconds.
✅ Preprocessing complete! Processed 110398 texts
⏱️  Preprocessing time: 289.49 seconds

⏳ Status: Starting LDA modeling (testing K=2 to K=32)...
   This will test 16 different K values - progress will be shown below:

--- 1. Creating Dictionary & Corpus for 'Negative' Model ---
Vocabulary size for 'Negative': 1941

--- 2. Finding Optimal K for 'Negative' Model (Sequentially) ---
Testing K=2...
K=2 Coherence: 0.3543
  K=2: Coherence=0.3543
Testing K=4...
K=4 C

In [ ]:
# ========== STEP 6: UPLOAD EVERYTHING TO GOOGLE DRIVE ==========
import os
import shutil
from google.colab import drive
from datetime import datetime

print("\n" + "="*80)
print("☁️  STEP 6: UPLOADING ALL RESULTS TO GOOGLE DRIVE")
print("="*80)

# Mount Google Drive
print("⏳ Mounting Google Drive...")
drive.mount('/content/drive', force_remount=True)
print("✅ Google Drive mounted!")

# Create output directory with timestamp
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
OUTPUT_DIR = f'/content/drive/MyDrive/LDA_Results_{timestamp}'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"📁 Created output directory: {OUTPUT_DIR}")

upload_start = datetime.now()

# List of all files to upload
local_files = [
    'lda_model_negative.model',
    'lda_model_negative.model.state',
    'lda_model_negative.model.id2word',
    'lda_model_negative.model.expElogbeta.npy',
    'dictionary_negative.dict',
    'corpus_negative.pkl',
    'processed_texts_negative.pkl',
    'all_models_negative.pkl',
    'lda_model_positive.model',
    'lda_model_positive.model.state',
    'lda_model_positive.model.id2word',
    'lda_model_positive.model.expElogbeta.npy',
    'dictionary_positive.dict',
    'corpus_positive.pkl',
    'processed_texts_positive.pkl',
    'all_models_positive.pkl',
    'lda_complete_results.txt',
    'lda_all_k_results_complete.json'
]

print("\n⏳ Uploading files to Google Drive...")
uploaded_count = 0
skipped_count = 0

for filename in local_files:
    if os.path.exists(filename):
        try:
            destination = os.path.join(OUTPUT_DIR, filename)
            shutil.copy2(filename, destination)
            uploaded_count += 1
            print(f"  ✓ Uploaded: {filename}")
        except Exception as e:
            print(f"  ✗ Failed to upload {filename}: {e}")
            skipped_count += 1
    else:
        print(f"  ⊘ Skipped (not found): {filename}")
        skipped_count += 1

upload_time = (datetime.now() - upload_start).total_seconds()

print("\n" + "="*80)
print("✅ UPLOAD COMPLETE!")
print("="*80)
print(f"\n📊 Upload Summary:")
print(f"  • Files uploaded: {uploaded_count}")
print(f"  • Files skipped: {skipped_count}")
print(f"  • Upload time: {upload_time:.2f} seconds")
print(f"\n📂 All files saved to:")
print(f"  {OUTPUT_DIR}")

# Create a shareable link instructions
print("\n" + "="*80)
print("📤 TO SHARE THESE FILES:")
print("="*80)
print("1. Go to Google Drive")
print(f"2. Navigate to: My Drive/LDA_Results_{timestamp}/")
print("3. Right-click the folder → 'Share' → 'Get link'")
print("4. Set to 'Anyone with the link can view'")
print("5. Copy and share the link!")
print("="*80 + "\n")

# Also create a README file in the folder
readme_content = f"""
LDA TOPIC MODELING RESULTS
Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

FILES INCLUDED:
===============

📖 HUMAN-READABLE FILES (START HERE):
- lda_complete_results.txt - Detailed report with best model topics
- lda_all_k_results_complete.json - ALL K values (2-32) with all topics

🤖 BINARY MODEL FILES (for reloading in Python):

Negative Sentiment:
- lda_model_negative.model - Best LDA model
- dictionary_negative.dict - Word dictionary
- corpus_negative.pkl - Document-term matrix
- processed_texts_negative.pkl - Preprocessed texts
- all_models_negative.pkl - All 16 models (K=2 to K=32)

Positive Sentiment:
- lda_model_positive.model - Best LDA model
- dictionary_positive.dict - Word dictionary
- corpus_positive.pkl - Document-term matrix
- processed_texts_positive.pkl - Preprocessed texts
- all_models_positive.pkl - All 16 models (K=2 to K=32)

HOW TO USE:
===========
1. Read lda_complete_results.txt for summary and best model details
2. Open lda_all_k_results_complete.json to see topics for ALL K values
3. Use binary .pkl files to reload models in Python/Colab

LOADING MODELS IN PYTHON:
=========================
from gensim.models import LdaModel
import pickle

# Load best negative model
model = LdaModel.load('lda_model_negative.model')

# Load all models
with open('all_models_negative.pkl', 'rb') as f:
    all_models = pickle.load(f)
"""

readme_path = os.path.join(OUTPUT_DIR, 'README.txt')
with open(readme_path, 'w', encoding='utf-8') as f:
    f.write(readme_content)
print("✅ Created README.txt with usage instructions")

print(f"\n🎉 ALL DONE! Your results are safely stored in Google Drive.")
print(f"📂 Folder: LDA_Results_{timestamp}")